# T45E / E11 — Chi phí phát hiện khi mô hình sinh cũng là mô hình đọc

**Câu hỏi.** Khi một mô hình vừa sinh câu trả lời vừa cung cấp trọng số chú ý (kịch bản tự kiểm tra),
việc phát hiện làm tốn thêm bao nhiêu thời gian? Trước đây mục này chỉ là lập luận, vì Qwen2.5-7B tràn số
ở `float16` nên không sinh được trên T4. Qwen2.5-3B không có lớp nào tràn số nên đảm nhận được cả hai vai.

**Cách đo.** 100 ngữ cảnh của tập test ViHallu (seed 42). Câu trả lời được sinh tự do một lần, rồi **phát
lại đúng chuỗi token đó** theo bốn cách, để mọi cách làm cùng một khối lượng tính toán:

| Cách | Cài đặt chú ý | Đọc chú ý | Ý nghĩa |
|---|---|---|---|
| A | SDPA | không | chi phí sinh hiện nay |
| B | eager | không | B − A: chi phí tắt cài đặt tối ưu |
| C | eager | hook, ngay khi sinh | C − B: chi phí của việc phát hiện; C − A: toàn bộ phần thêm |
| R | eager | hook, đọc lại một lượt | chi phí nếu phải đọc lại sau khi sinh |

Mỗi cách chạy hai lượt mỗi mẫu, thứ tự xoay vòng và đảo ngược, để nếu card chậm dần thì chậm đều cả bốn.

**Dự đoán ghi trước khi chạy** nằm trong `PREDICTIONS` của `scripts/measure_self_check.py`.

**Chạy trong một phiên T4 riêng.** Ước tính 1,5–2 giờ. Bị ngắt giữa chừng thì chạy lại ô 6: các mẫu đã đo
không đo lại. Dùng *Save Version* (Save & Run All) là đủ.

In [1]:
# Ô 1 — lấy mã nguồn. Chạy lại được nhiều lần.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

/kaggle/working/vihallulens
commit: e7f0335 Merge pull request #21 from wsunicorn/t45e-chi-phi-tu-kiem-tra


In [2]:
# Ô 2 — cài đặt, ghim phiên bản theo uv.lock.
!pip install -q --no-deps -e .
!pip install -q -r requirements-kaggle.txt

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for vihallulens (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vihallulens 0.1.0 requires pyvi, which is not installed.


In [3]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
import importlib.util
import sys
from pathlib import Path

import torch

sys.path.insert(0, "src")
from vihallulens.data.paths import find_raw_dir

problems = []
absent = [n for n in ("torch", "transformers", "bitsandbytes", "accelerate", "pandas")
          if importlib.util.find_spec(n) is None]
if absent:
    problems.append(f"thiếu gói {absent}")
if not torch.cuda.is_available():
    problems.append("chưa bật GPU T4")
else:
    print("GPU:", torch.cuda.get_device_name(0))
try:
    raw = Path(find_raw_dir())
    missing = [n for n in ['vihallu_train.csv'] if not (raw / n).is_file()]
    print("dữ liệu thô:", raw, "— thiếu", missing or "không")
    if missing:
        problems.append(f"thiếu tệp {missing}")
except Exception as error:
    problems.append(f"chưa gắn dataset dữ liệu thô ({error})")
if problems:
    raise SystemExit("TIỀN KIỂM HỎNG: " + "; ".join(problems))
print("Tiền kiểm đạt.")

GPU: Tesla T4
dữ liệu thô: /kaggle/input/datasets/unicorn1209/vihallulens — thiếu không
Tiền kiểm đạt.


In [4]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_incremental.py tests/test_self_check.py tests/test_revisions.py -q


MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : e7f0335 Merge pull request #21 from wsunicorn/t45e-chi-phi-tu-kiem-tra
  python           : 3.13.16
  torch            : 2.11.0+cu130
  transformers     : 5.17.0
  bitsandbytes     : 0.50.2
  accelerate       : 1.15.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv

CHUẨN HÓA VIHALLU
  nguồn                 : /kaggle/input/datasets/unicorn1209/vihallulens
  số dòng               : 7,000
  ngữ cảnh d

In [5]:
# Ô 5 — chạy thử 2 mẫu. Khoảng 5 phút gồm tải mô hình. Đọc dòng "thử float16": logit phải hữu hạn
# và câu trả lời phải là tiếng Việt bình thường. Nếu float16 hỏng, script tự chuyển sang bfloat16.
!python scripts/measure_self_check.py run --limit 2 --out /kaggle/working/e11_self_check.jsonl
!python scripts/measure_self_check.py report --path /kaggle/working/e11_self_check.jsonl

config.json: 100%|█████████████████████████████| 661/661 [00:00<00:00, 3.51MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 29.8MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 47.9MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 125MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 154MB/s]
model.safetensors.index.json: 100%|████████| 35.6k/35.6k [00:00<00:00, 94.4MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  0.00B            

Fetching 2 files:   0%|                                   | 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0%|        |  0.00B / 6.17GB            
Reconstructing (incomplete total...):   0%|        |  0.00B / 6.17GB            
Reconstructing (incomplete total...):  28%|██▏     | 1.73GB / 6.17GB,  420MB/s  
Reconstructing (incomplete total...):  34%|██▋     | 2.10GB / 6.17GB,  863MB/s  
Reconstructing (incomplete 

In [6]:
# Ô 6 — đủ 100 mẫu. In tiến độ mỗi 5 mẫu.
!python scripts/measure_self_check.py run --out /kaggle/working/e11_self_check.jsonl

Loading weights: 100%|████████████████████████| 434/434 [00:05<00:00, 75.91it/s]
  thử float16: logit hữu hạn = True | 'Theo ngữ cảnh được cung cấp, mật độ dân số cao nhất của tỉnh Khánh Hòa là huyện đảo Trường'
  thử float16: logit hữu hạn = True | 'Theo ngữ cảnh được cung cấp, không có thông tin nào cho thấy Birmingham đã phát triển mạng'
  thử float16: logit hữu hạn = True | 'Theo ngữ cảnh được cung cấp, sự kiện diễn ra vào cuối tháng 10 năm 1740 khi Hoàng đế Karl '
  5/100 mẫu
  10/100 mẫu
  15/100 mẫu
  20/100 mẫu
  25/100 mẫu
  30/100 mẫu
  35/100 mẫu
  40/100 mẫu
  45/100 mẫu
  50/100 mẫu
  55/100 mẫu
  60/100 mẫu
  65/100 mẫu
  70/100 mẫu
  75/100 mẫu
  80/100 mẫu
  85/100 mẫu
  90/100 mẫu
  95/100 mẫu
  100/100 mẫu


In [7]:
# Ô 7 — tổng hợp. Tải về hai tệp trong /kaggle/working: e11_self_check.jsonl và e11_self_check.json.
!python scripts/measure_self_check.py report --path /kaggle/working/e11_self_check.jsonl --json /kaggle/working/e11_self_check.json

Qwen/Qwen2.5-3B-Instruct · float16 · Tesla T4 · transformers 5.17.0
100 mẫu, bỏ 0; trung bình 297 token prompt, 94.5 token phản hồi
  A (sdpa )    6569.8 ms/mẫu · đỉnh 2,088 MB · lượt 2 so lượt 1 -0.3%
  B (eager)    6414.8 ms/mẫu · đỉnh 2,099 MB · lượt 2 so lượt 1 +0.1%
  C (eager)    7751.2 ms/mẫu · đỉnh 2,090 MB · lượt 2 so lượt 1 -0.2%
  R (eager)     239.1 ms/mẫu · đỉnh 2,083 MB · lượt 2 so lượt 1 +0.1%
  C-B    1336.4 ms [1209.8; 1458.3] = 20.3% thời gian sinh A
  B-A    -155.0 ms [-172.4; -138.6] = -2.4% thời gian sinh A
  C-A    1181.3 ms [1065.5; 1290.4] = 18.0% thời gian sinh A
  R       239.1 ms [230.3; 248.4] = 3.6% thời gian sinh A
  tỷ lệ token trùng khi phát lại: {'A': 1.0, 'B': 0.9982002964217658, 'C': 0.9982002964217658}
  |Δ| C–R của lookback_total: từng token 0.00065; sau khi lấy trung bình 0.00022 (lớn nhất 0.00626)
  dự đoán «C − B dưới 10 % thời gian sinh A»: SAI
  dự đoán «B − A dương: tắt sdpa làm sinh chậm hơn»: SAI
  dự đoán «C − A nhỏ hơn R: đọc khi sinh ít t